<a href="https://colab.research.google.com/github/sagnikdas1610/bioinformatics-3d-project/blob/main/AI_Structure_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title AI 3D Protein Structure Generator (ESMFold API — Validated Single Chain)
#@markdown Paste a single-chain amino acid sequence (10-400 standard residues) and run this cell to fold it via the ESM Atlas API and download the predicted `.pdb` file.

import re
import requests
from google.colab import files

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
SEQUENCE = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG" #@param {type:"string"}
OUTPUT_FILENAME = "esmfold_predicted_protein.pdb" #@param {type:"string"}

clean_seq = re.sub(r"\s+", "", SEQUENCE).upper()
invalid = sorted(set(clean_seq) - STANDARD_AA)
if not clean_seq or invalid:
    raise ValueError(f"Invalid amino-acid symbols found: {invalid}. Use standard 20 codes (ACDEFGHIKLMNPQRSTVWY).")
if not (10 <= len(clean_seq) <= 400):
    raise ValueError(f"Sequence length ({len(clean_seq)} aa) must be between 10 and 400 residues for the public ESMFold API. Use ColabFold for longer sequences.")

api_url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
print(f"Submitting sequence ({len(clean_seq)} residues) to ESMFold Atlas API...")

response = requests.post(api_url, data=clean_seq, timeout=60)
if response.status_code != 200 or "ATOM  " not in response.text:
    raise RuntimeError(
        f"ESMFold API returned HTTP {response.status_code}. Public servers may be rate-limited or busy. "
        "No fallback structure was substituted; try ColabFold or AlphaFold DB instead."
    )

pdb_data = response.text
ca_b = [float(line[60:66]) for line in pdb_data.splitlines() if line.startswith("ATOM  ") and line[12:16].strip() == "CA"]
if ca_b:
    scale = 100.0 if max(ca_b) <= 1.05 else 1.0
    mean_plddt = sum(v * scale for v in ca_b) / len(ca_b)
    print(f"Prediction succeeded. Mean C-alpha pLDDT: {mean_plddt:.1f} / 100")

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    f.write(pdb_data)

print(f"Downloading {OUTPUT_FILENAME}...")
files.download(OUTPUT_FILENAME)
